# Lab 2 exercise — MCP date server + client

I’m working through the Week 6 Lab 2 exercise from `6_mcp/2_lab2.ipynb`: build my own MCP server, expose a simple tool, then hook it up to an agent and (optionally) to a plain OpenAI client.

**What I set out to do**

- Implement a small **MCP server** (`date_server.py`) with one tool that returns today’s date.
- Prove it works with the **OpenAI Agents SDK** using `MCPServerStdio` and an `Agent`, the same style as the accounts lab.
- Add **`date_client.py`** so I can list and call tools over stdio without going through the high-level agent first — mirroring `accounts_client.py`.
- Go one step further: use **OpenAI Chat Completions + a tool loop** directly, without `Agent` / `Runner`, so I understand what the SDK is abstracting.

**What I’m trying to get out of this**

A concrete feel for how MCP servers register tools, how clients discover them, and how a model actually triggers a tool call end-to-end.

I’m keeping this under `community_contributions/` so it’s easy to open as a single PR. The first code cell resolves paths; my kernel usually works best from the repo root (`agents/`) or from this folder.

### Code cell 1 — Find `date_server.py` and the repo root

**What I’m doing here:** Figuring out `EXERCISE_DIR` (this folder) and `REPO_ROOT` (the `agents` project root).

**Why it matters:** `uv run` needs the repo root as `cwd` so it picks up the course `pyproject.toml` and dependencies like `mcp`. The server script itself stays in this exercise folder.

In [ ]:
from pathlib import Path


def resolve_exercise_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "6_mcp/community_contributions/mcp_lab2_date_exercise",
        cwd,
        cwd.parent / "mcp_lab2_date_exercise",
    ]
    for c in candidates:
        if (c / "date_server.py").exists():
            return c.resolve()
    raise FileNotFoundError(
        "Could not find date_server.py. Open the repo root or this folder as Jupyter cwd, then restart the kernel."
    )


EXERCISE_DIR = resolve_exercise_dir()
REPO_ROOT = EXERCISE_DIR.parent.parent.parent

print("EXERCISE_DIR =", EXERCISE_DIR)
print("REPO_ROOT   =", REPO_ROOT)

### Code cell 2 — Imports (Agents SDK path)

**What I’m doing here:** Loading `.env` and importing the same MCP + Agent pieces as in `2_lab2.ipynb`.

**Why it matters:** I’m following the course pattern: spawn an MCP server over stdio, then drive it with `Agent` + `Runner`.

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio

load_dotenv(override=True)

### Code cell 3 — List tools from the date MCP server

**What I’m doing here:** Starting `date_server.py` with `uv run`, connecting over stdio, and calling `list_tools()`.

**Why it matters:** I want to see `get_current_date` show up before I add the model — isolates “server works” from “LLM works.”

In [ ]:
date_mcp_params = {
    "command": "uv",
    "args": ["run", str(EXERCISE_DIR / "date_server.py")],
    "cwd": str(REPO_ROOT),
}

async with MCPServerStdio(params=date_mcp_params, client_session_timeout_seconds=30) as server:
    date_tools = await server.list_tools()

date_tools

### Code cell 4 — Agent asks for today’s date (Agents SDK)

**What I’m doing here:** Attaching my MCP server to an `Agent` and sending a single user message under a trace.

**Why it matters:** Same story as the accounts lab — I want to watch the model choose the tool and use the returned value.

In [ ]:
instructions = "You can use tools to answer questions. Be concise."
request = "What is today's date? Use the available tool to get it."
model = "gpt-4o-mini"

async with MCPServerStdio(params=date_mcp_params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(
        name="date_helper",
        instructions=instructions,
        model=model,
        mcp_servers=[mcp_server],
    )
    with trace("date_mcp_agent"):
        result = await Runner.run(agent, request)
    print(result.final_output)

### Code cell 5 — MCP client module (like `accounts_client.py`)

**What I’m doing here:** Importing my `date_client.py` and printing what `list_date_tools()` returns.

**Why it matters:** I wanted the same shape as `accounts_client.py` — raw stdio + `ClientSession` — without the Agents SDK in the middle.

In [ ]:
import sys

sys.path.insert(0, str(EXERCISE_DIR))

from date_client import list_date_tools

tools = await list_date_tools()
for t in tools:
    print(t.name, "—", (t.description or "").split("\n")[0])

### Code cell 6 — OpenAI Chat Completions only (no Agents SDK)

**What I’m doing here:**

- Holding one MCP session open (`date_server_session`).
- Turning MCP tool definitions into OpenAI `tools` JSON.
- Looping: if the model returns `tool_calls`, I run `session.call_tool` myself and append `role: tool` messages until I get a normal answer.

**Why it matters:** This was the stretch goal for me — same outcome as the agent cell, but I’m driving **Chat Completions** and the tool loop by hand so I see what `Runner` usually hides.

In [ ]:
import json

from openai import AsyncOpenAI

from date_client import date_server_session, mcp_tools_to_openai_functions


def _tool_result_text(result) -> str:
    parts = []
    for block in result.content:
        if getattr(block, "text", None):
            parts.append(block.text)
    return "\n".join(parts) if parts else str(result)


async def ask_openai_for_todays_date() -> None:
    client = AsyncOpenAI()

    async with date_server_session() as session:
        listed = await session.list_tools()
        openai_tools = mcp_tools_to_openai_functions(listed.tools)

        messages = [
            {
                "role": "user",
                "content": "What is today's date? You must call the tool to get it.",
            }
        ]

        for _ in range(6):
            completion = await client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
                tools=openai_tools,
            )
            choice = completion.choices[0]
            msg = choice.message

            if not msg.tool_calls:
                print("Final:", msg.content)
                return

            messages.append(msg.model_dump())

            for tc in msg.tool_calls:
                name = tc.function.name
                raw_args = tc.function.arguments or "{}"
                args = json.loads(raw_args)
                tr = await session.call_tool(name, args)
                text = _tool_result_text(tr)
                messages.append(
                    {"role": "tool", "tool_call_id": tc.id, "content": text}
                )

        print("Stopped after max tool rounds.")


await ask_openai_for_todays_date()

### Trace (optional)

After I run the agent cell, I sometimes check [OpenAI traces](https://platform.openai.com/traces) for `date_mcp_agent` to see the run spelled out.